# 01c PyOptInterface

[PyOptInterface](https://metab0t.github.io/PyOptInterface/) is a thin, fast modelling layer over
several solvers. Its PIPS-IPM++ backend is `pyoptinterface.pipsipmpp`, and the block of a variable
is simply an argument on the variable, the most direct mapping onto what the solver wants. It is
also the fastest of the frameworks at building a model, which matters once a model has millions
of variables.

This notebook needs an optional install, because the backend is a fork that compiles from source:

```console
$ uv run --extra pyoptinterface jupyter lab      # needs a C++ compiler on your machine
$ pixi run -e pyoptinterface lab                 # pixi brings its own compiler
```

On Linux add `--extra solver` to the uv command to solve as well; the pixi environment already
includes it. Without the install the cells are skipped, and [01a linopy](01a-linopy.ipynb) is the
closest alternative: its ideas carry over one to one.

| sections | run on |
|---|---|
| 1 to 3: build, export, visualize, change the model, read the structure back | every machine with the `pyoptinterface` extra |
| 4: solve with PIPS-IPM++ | Linux or WSL with the `solver` extra as well |
| 5: more than one process with MPI | Linux or WSL with the `solver` extra as well |

In [ ]:
import workshop

workshop.check()

<details><summary>💡 Hint: <code>pyoptinterface  not installed</code></summary>

JupyterLab was started from an environment without the extra. Stop it and start it again with
`uv run --extra pyoptinterface jupyter lab` or `pixi run -e pyoptinterface lab`. With uv, a plain
`uv sync` removes the extra again, so pass `--extra pyoptinterface` whenever you sync.
</details>

## 1. The model

The sample model from [00](00-start-here.ipynb). The data (costs, profiles, the emission budget)
lives in `sample_data.py` next to this notebook, so every framework computes the same numbers.

The cell below writes the model into a file, `poi_model.py`, instead of running it. That way this
notebook and the parallel run in section 5 use the very same code. The cell after it loads the
file.

The annotation is the `block=` argument of every `add_variable`: `0` for the root, which is also
the default, and `1 + t // hours` for the leaf of hour `t`.

In [ ]:
%%writefile poi_model.py
import pyoptinterface as poi
from pyoptinterface import pipsipmpp

from sample_data import STORAGE, TECHS, budget, profiles


def build(T=24, n_blocks=4, techs=TECHS, storage=False, **model_args):
    demand, avail = profiles(T)
    hours = -(-T // n_blocks)                                     # hours per block, rounded up
    model = pipsipmpp.Model(**model_args)
    add = model.add_variable

    # block=0 is the root and the default; the leaf of hour t is 1 + t // hours
    v = {("cap", g): add(lb=0.0, name=f"cap[{g}]", block=0) for g in techs}
    for t in range(T):
        for g in techs:
            v["gen", g, t] = add(lb=0.0, name=f"gen[{g},{t}]", block=1 + t // hours)
        if storage:
            for name in ("charge", "discharge", "soc"):
                v[name, t] = add(lb=0.0, name=f"{name}[{t}]", block=1 + t // hours)
    if storage:
        v["sto_cap"] = add(lb=0.0, name="sto_cap", block=0)

    cons = {}
    for t in range(T):
        supply = poi.quicksum(v["gen", g, t] for g in techs)
        if storage:
            supply += v["discharge", t] - v["charge", t]
            # the level carries over from the previous hour; the first hour starts empty
            previous = v["soc", t - 1] if t > 0 else 0.0
            cons["soc", t] = model.add_linear_constraint(
                v["soc", t] - previous - STORAGE["efficiency"] * v["charge", t]
                + v["discharge", t], poi.Eq, 0.0, name=f"soc[{t}]")
            cons["soc_limit", t] = model.add_linear_constraint(
                v["soc", t] - v["sto_cap"], poi.Leq, 0.0, name=f"soc_limit[{t}]")
        cons["balance", t] = model.add_linear_constraint(
            supply, poi.Eq, demand[t], name=f"balance[{t}]")
        for g in techs:
            cons["limit", g, t] = model.add_linear_constraint(
                v["gen", g, t] - avail[techs[g][3]][t] * v["cap", g], poi.Leq, 0.0,
                name=f"limit[{g},{t}]")
    # linking; a zero coefficient would be stored as an entry, so emission-free ones are left out
    cons["carbon"] = model.add_linear_constraint(
        poi.quicksum(techs[g][2] * v["gen", g, t] for g in techs if techs[g][2]
                     for t in range(T)),
        poi.Leq, budget(T), name="carbon")

    cost = (poi.quicksum(techs[g][0] * v["cap", g] for g in techs)
            + poi.quicksum(techs[g][1] * v["gen", g, t] for g in techs for t in range(T)))
    if storage:
        cost += STORAGE["investment"] * v["sto_cap"]
    model.set_objective(cost, poi.ObjectiveSense.Minimize)
    model.set_raw_parameter("SILENT", True)
    return model, v, cons

In [ ]:
%%requires pyoptinterface
%run poi_model.py
import pyoptinterface as poi

model, v, cons = build()
print(len(v), "variables,", len(cons), "constraints")

<details><summary>💡 Hint: I changed the model. Why did nothing happen?</summary>

`%%writefile` saves the code and `%run` loads it. After editing the first cell, run both again,
then `build()` gives you the new model.
</details>

## 2. Export it and look at the matrix

`model.write_parquet` writes the model the way PIPS-IPM++ reads it: the matrix, the bounds and
costs, and the block of every row and column. It does not need the solver or MPI.

- `layout="monolithic"` writes one set of files for the whole model. (`"distributed"` writes one
  set per block, see [04](04-deep-dive.ipynb).)
- `names=True` keeps the names given to `add_variable` and `add_linear_constraint`.

In [ ]:
%%requires pyoptinterface
from pathlib import Path

Path("data").mkdir(exist_ok=True)


def export(model, name):
    return model.write_parquet(f"data/{name}", layout="monolithic", names=True)


stem = export(model, "poi_base")
workshop.visualize(stem)

`workshop.visualize` runs `pipstools visualize` on the files, shows the picture, and prints the
numbers that decide whether a structure is good. How to read it:

- Rows and columns are sorted by block. Each dot is a nonzero.
- The narrow band on the left holds the 2 root columns, `cap[wind]` and `cap[gas]`. Its dots are
  the capacity limits of every hour, which use the capacity of their technology.
- The four coloured squares are the leaf blocks, six hours each: 12 `gen` columns and 18 rows
  (6 balance, 12 limit).
- The grey row at the bottom is the one linking row, the carbon budget. It touches every leaf.

This is the structure from the table in 00, now drawn from the actual matrix.

### Read the structure back, without any files

PyOptInterface offers something the other frameworks do not: the block is a first-class
attribute, so you can ask the model how it classified every constraint.

In [ ]:
%%requires pyoptinterface
from collections import Counter


def constraint_blocks(model, cons):
    # 0 = root, k = local to leaf k, -1 = linking
    return Counter(model.get_constraint_attribute(c, poi.ConstraintAttribute.Block)
                   for c in cons.values())


print("constraints per block:", dict(sorted(constraint_blocks(model, cons).items())))
block = lambda c: model.get_constraint_attribute(c, poi.ConstraintAttribute.Block)
print("balance[0] ->", block(cons["balance", 0]), "| limit[gas,23] ->",
      block(cons["limit", "gas", 23]), "| carbon ->", block(cons["carbon"]))

One constraint with block `-1`: the carbon budget, linking. Counting the `-1`s is the quickest
check of an annotation from inside a script.

## 3. Change the model and watch the matrix

Each exercise changes the model once. Before you run a cell, predict what happens to the three
numbers: root columns, columns per leaf, linking rows. Then compare with the picture.

### Exercise 1: more hours

Two days instead of one, still in 4 blocks, and then in 8.

In [ ]:
%%requires pyoptinterface
workshop.visualize(export(build(T=48)[0], "poi_T48"))

In [ ]:
%%requires pyoptinterface
workshop.visualize(export(build(T=48, n_blocks=8)[0], "poi_T48_8b"))

The root stays at 2 columns however many hours you add, and there is still one linking row. More
hours make each leaf larger (with 4 blocks) or give more leaves (with 8). This is the property
that makes energy models a good fit: they grow along time, and time goes into the leaves.

Try `T=168`, a week, with 7 blocks of one day each.

### Exercise 2: another technology

`sample_data.py` describes a technology as one row: investment cost, variable cost, emissions and
availability profile. `SOLAR` is a row prepared for you; adding it is one line.

In [ ]:
%%requires pyoptinterface
from sample_data import SOLAR, TECHS

with_solar = TECHS | {"solar": SOLAR}
print(with_solar)
workshop.visualize(export(build(techs=with_solar)[0], "poi_solar"))

One more root column, `cap[solar]`, and every leaf grows by one `gen` column and one limit row
per hour. The linking row stays alone.

<details><summary>💡 Hint: 240 nonzeros here, but 228 in the other 01 notebooks?</summary>

Solar is not available at night, so its limit rows read `gen[solar,t] - 0 * cap[solar] <= 0` for
twelve hours. linopy, Pyomo and GAMS drop a coefficient of 0; PyOptInterface keeps it as an entry
of the matrix. It changes nothing about the structure or the answer. The model file leaves the
zero emissions out of the carbon row for the same reason.
</details>

Invent your own technology: a tuple `(investment, variable, emission, profile)` with one of the
profiles `"wind"`, `"solar"` or `"firm"`. A cheap, dirty one such as `(0.2, 0.8, 0.9, "firm")`
changes the answer a lot and the structure not at all.

### Exercise 3: storage

`build(storage=True)` adds the storage from 00. Read the `if storage:` parts of the model first.
The constraint to look at is the level `soc[t]`, which uses `soc[t - 1]`. Inside a block that row
is local. In the first hour of a block it is not. Predict the number of linking rows, then run:

In [ ]:
%%requires pyoptinterface
model, v, cons = build(storage=True)
print("constraints per block:", dict(sorted(constraint_blocks(model, cons).items())))
workshop.visualize(export(model, "poi_storage"))

Four linking rows: the carbon budget, plus one storage level per block boundary. Those three are
2-link rows: each touches only two neighbouring blocks, and you can see them in the picture as
short strokes at the bottom, right at the corners between two blocks. PIPS-IPM++ handles them
much more cheaply than the carbon budget, which touches every block.

The storage capacity `sto_cap` is the third root column.

### Exercise 4: change the blocks afterwards

The block is an attribute, so it can be changed on a model that is already built. Here every
fourth hour goes to the same block. A perfectly valid split, and a very poor one for a model with
storage:

In [ ]:
%%requires pyoptinterface
model, v, cons = build(storage=True)
for key, var in v.items():
    if key[0] in ("gen", "charge", "discharge", "soc"):         # the variables with an hour
        hour = key[-1]
        model.set_variable_attribute(var, poi.VariableAttribute.Block, 1 + hour % 4)

print("constraints per block:", dict(sorted(constraint_blocks(model, cons).items())))
workshop.visualize(export(model, "poi_roundrobin"))

Now every storage level links two blocks, and the bottom border is as tall as a leaf. Most of
those rows are still 2-link (block 1 to 2, 2 to 3, 3 to 4). The ones from block 4 back to block 1
are not, because blocks 4 and 1 are not neighbours.

The model is the same and its answer is the same. Only the split changed, and with it how much
work the borders cost.

### Exercise 5: your own change

Some ideas, each with a prediction to check against the picture:

- Make the storage cyclic, so the last hour feeds the first: set `previous` to
  `v["soc", T - 1]` in the first hour instead of `0.0`, run both cells of section 1 again, and
  export `build(storage=True)[0]`.
- Split into regions or scenarios instead of hours: add an index, then decide the block by it.
- Blocks of different lengths: change the rule `1 + t // hours`.

<details><summary>💡 Answer: what does the cyclic storage do?</summary>

It adds one linking row that ties the last hour of block 4 to the first hour of block 1. That row
touches two blocks, but not two neighbouring ones, so it is not 2-link: the summary reports 5
linking rows, of which 3 are 2-link. One such row is harmless. A model with many of them is worth
reformulating.
</details>

## 4. Solve with PIPS-IPM++

From here on the cells need the solver as well: Linux or WSL, with JupyterLab started through
`uv run --extra pyoptinterface --extra solver jupyter lab` or `pixi run -e pyoptinterface lab`.
On other machines they are skipped.

For reference, HiGHS finds the objective `2.744656` for the base model and `2.121267` with
storage (see section 4 of the other 01 notebooks).

In [ ]:
%%requires pyoptinterface solver
model, v, cons = build()
model.optimize()

In [ ]:
%%requires pyoptinterface solver
print("status    ", model.get_model_attribute(poi.ModelAttribute.TerminationStatus))
print("objective ", round(model.get_model_attribute(poi.ModelAttribute.ObjectiveValue), 6))
print("capacities", {g: round(model.get_value(v["cap", g]), 4) for g in TECHS})

The same answer as HiGHS, to about six figures. PIPS-IPM++ is an interior-point method and stops
within a tolerance of the optimum, so the last digits may differ.

The solve cell shows what PIPS-IPM++ prints: its parameters, a presolve summary and the residuals
at the end. `SILENT`, set in the model file, removes most of the iteration log but not those,
which is why the results get a cell of their own.

### The split changes the work, never the answer

In [ ]:
%%requires pyoptinterface solver
objective = lambda m: m.get_model_attribute(poi.ModelAttribute.ObjectiveValue)
splits = {}
for n_blocks in (2, 4, 6, 12):
    model, *_ = build(storage=True, n_blocks=n_blocks)
    model.optimize()
    splits[f"{n_blocks} contiguous blocks"] = objective(model)

model, v, _ = build(storage=True)
for key, var in v.items():
    if key[0] in ("gen", "charge", "discharge", "soc"):
        model.set_variable_attribute(var, poi.VariableAttribute.Block, 1 + key[-1] % 4)
model.optimize()
splits["round robin"] = objective(model)

In [ ]:
%%requires pyoptinterface solver
for label, value in splits.items():
    print(f"{label:20s} objective {value:.6f}")
print(f"{'HiGHS':20s} objective 2.121267")

If changing the split ever changes your objective beyond the solver tolerance, the annotation has
a bug.

### Options

```python
model = pipsipmpp.Model(options_file="tuned.opt")    # a base configuration
model.set_raw_parameter("SCALER", "none")            # overrides the file
model.optimize(options={"REGULARIZATION": True})     # and so does this
```

`get_raw_parameter(name)` reads an option back; `Model(comm=...)` picks the communicator, by
default `MPI.COMM_WORLD`.

## 5. More than one process

A notebook is a single process, so everything above solved on one. To use the decomposition, the
same code runs under MPI as a script. Every process runs the whole script: building a
PyOptInterface model is cheap, and `optimize()` uses the model from process 0 only. `optimize()`
is collective, so every process must reach it.

In [ ]:
%%writefile poi_parallel.py
import pyoptinterface as poi
from mpi4py import MPI
from poi_model import build

comm = MPI.COMM_WORLD
model, *_ = build(storage=True, comm=comm)
model.optimize()
if comm.Get_rank() == 0:
    print(f"ranks={comm.Get_size()} "
          f"{model.get_model_attribute(poi.ModelAttribute.TerminationStatus)} "
          f"objective={model.get_model_attribute(poi.ModelAttribute.ObjectiveValue):.6f}")

In [ ]:
%%requires pyoptinterface solver
for ranks in (1, 2, 4):
    workshop.mpirun("poi_parallel.py", ranks, show="ranks=")

The same objective on one, two and four processes, within the solver tolerance. The work is
split differently, so the last digits can differ, and so can the number of iterations.

`workshop.mpirun` runs `mpiexec -n 4 python poi_parallel.py` for you; in a terminal you can type that yourself.

<details><summary>💡 Hint: how many processes should I use?</summary>

At most one per leaf block, since PIPS-IPM++ refuses more processes than leaves. Fewer is fine;
leaves then share a process. On a laptop, the number of CPU cores is the useful upper limit.
</details>

## 6. Now your own model

PyOptInterface fits well when you generate a model programmatically rather than from labelled
arrays, when build time matters, or when you want to check the structure in your own code:

```python
x = {(r, t): model.add_variable(lb=0.0, block=block_of(r, t)) for r in regions for t in hours}
...
linking = [c for c in constraints
           if model.get_constraint_attribute(c, poi.ConstraintAttribute.Block) == -1]
print(f"{len(linking)} linking constraints out of {len(constraints)}")
```

If that number is large, rethink the split before you queue a cluster job.

## 7. Checkpoint

- Why is `block=0` a sensible default for a variable you say nothing about?
- What does a constraint block of `-1` mean, and what makes there be many of them?

## Next

[03 Structure detection](03-structure-detection.ipynb), which finds a structure in a model that
states none. Or [02 PyPSA](02-pypsa.ipynb) if you use PyPSA.

## Notes

- Continuous, linear variables only. A non-continuous domain raises `ValueError`.
- Constraint senses are `Eq`, `Leq` and `Geq`; write a range as two constraints.
- Maximisation is handled for you: objective and duals are flipped back on the way out.